In [ ]:
!pip install -q python-telegram-bot
!pip install -q openai-whisper
!pip install -q ffmpeg-python
!pip install -q requests
!pip install -q nest_asyncio

!sudo apt-get update -y
!sudo apt-get install -y ffmpeg

In [ ]:
import requests
import whisper
import subprocess
import nest_asyncio

from telegram import Update
from telegram.ext import (
    ApplicationBuilder,
    MessageHandler,
    ContextTypes,
    filters
)

In [ ]:
TELEGRAM_TOKEN = "8666772581:AAGAry2BCJbqmvDjx9FnbIgKO1ypIvnglkU"

TMDB_API_KEY = "fa152016b9257be119f3d756d6388103"

In [ ]:
print("Carregando Whisper...")

whisper_model = whisper.load_model("base")

print("Whisper carregado")

In [ ]:
BASE_URL = "https://api.themoviedb.org/3"


def get_genres():

    url = f"{BASE_URL}/genre/movie/list"

    params = {
        "api_key": TMDB_API_KEY,
        "language": "pt-BR"
    }

    response = requests.get(
        url,
        params=params
    )

    data = response.json()

    genres = {}

    for genre in data["genres"]:

        genres[
            genre["name"].lower()
        ] = genre["id"]

    return genres


GENRES = get_genres()


def search_movie(movie_name):

    url = f"{BASE_URL}/search/movie"

    params = {
        "api_key": TMDB_API_KEY,
        "query": movie_name,
        "language": "pt-BR"
    }

    response = requests.get(
        url,
        params=params
    )

    data = response.json()

    if not data["results"]:
        return None

    return data["results"][0]


def get_popular_movies():

    url = f"{BASE_URL}/movie/popular"

    params = {
        "api_key": TMDB_API_KEY,
        "language": "pt-BR"
    }

    response = requests.get(
        url,
        params=params
    )

    return response.json()["results"][:5]


def get_movies_by_genre(genre_id):

    url = f"{BASE_URL}/discover/movie"

    params = {
        "api_key": TMDB_API_KEY,
        "with_genres": genre_id,
        "language": "pt-BR",
        "sort_by": "vote_average.desc",
        "vote_count.gte": 500
    }

    response = requests.get(
        url,
        params=params
    )

    return response.json()["results"][:5]


def count_movies_by_genre(genre_id):

    url = f"{BASE_URL}/discover/movie"

    params = {
        "api_key": TMDB_API_KEY,
        "with_genres": genre_id,
        "language": "pt-BR"
    }

    response = requests.get(
        url,
        params=params
    )

    data = response.json()

    return data["total_results"]


print("TMDB configurado")
print(GENRES)

In [ ]:
def clean_movie_name(text):

    text = text.lower()

    remove_words = [
        "qual a nota de",
        "nota de",
        "me conte sobre",
        "me fale sobre",
        "descreva",
        "descreva o filme",
        "descreva a série",
        "filmes de",
        "filme de",
        "me recomende",
        "quais são os melhores",
        "quantos",
        "quantidade de",
        "existem",
        "filme",
        "filmes",
        "serie",
        "série",
        "sobre",
        "?",
    ]

    for word in remove_words:
        text = text.replace(word, "")

    return text.strip()


def generate_response(question):

    question_lower = question.lower()

    for genre_name, genre_id in GENRES.items():

        if (
            genre_name in question_lower
            and (
                "quantos" in question_lower
                or "quantidade" in question_lower
            )
        ):

            total = count_movies_by_genre(
                genre_id
            )

            return (
                f"Existem aproximadamente "
                f"{total} filmes de {genre_name}."
            )

    if (
        "menos populares" in question_lower
        or "menos famoso" in question_lower
        or "menos conhecidos" in question_lower
    ):

        movies = get_unpopular_movies()

        response = (
            "Filmes menos populares:\n\n"
        )

        for movie in movies:

            response += (
                f"{movie['title']} "
                f"{movie['vote_average']}\n"
            )

        return response

    for genre_name, genre_id in GENRES.items():

        if genre_name in question_lower:

            movies = get_movies_by_genre(
                genre_id
            )

            response = (
                f"Filmes de {genre_name}:\n\n"
            )

            for movie in movies:

                response += (
                    f"{movie['title']} "
                    f"{movie['vote_average']}\n"
                )

            return response

    if "popular" in question_lower:

        movies = get_popular_movies()

        response = "Filmes populares:\n\n"

        for movie in movies:

            response += (
                f"{movie['title']} "
                f"{movie['vote_average']}\n"
            )

        return response

    movie_name = clean_movie_name(
        question
    )

    movie = search_movie(
        movie_name
    )

    if not movie:
        return "Não encontrei esse filme ou série."

    title = movie["title"]

    rating = movie["vote_average"]

    release = movie.get(
        "release_date",
        "Desconhecida"
    )

    overview = movie.get(
        "overview",
        "Sem descrição."
    )


    if (
        "nota" in question_lower
        or "avaliação" in question_lower
        or "rating" in question_lower
    ):

        return (
            f"A nota de {title} é {rating}."
        )

    if (
        "descreva" in question_lower
        or "me conte" in question_lower
        or "me fale" in question_lower
        or "sobre" in question_lower
    ):

        return (
            f"{title}\n\n"
            f"Nota: {rating}\n"
            f"Lançamento: {release}\n\n"
            f"{overview}"
        )

    return (
        f"{title}\n"
        f"{rating}"
    )

In [ ]:
async def handle_text(
    update: Update,
    context: ContextTypes.DEFAULT_TYPE
):

    question = update.message.text

    response = generate_response(
        question
    )

    await update.message.reply_text(
        response
    )

In [ ]:
async def handle_voice(
    update: Update,
    context: ContextTypes.DEFAULT_TYPE
):

    voice = update.message.voice

    file = await context.bot.get_file(
        voice.file_id
    )

    await file.download_to_drive(
        "audio.ogg"
    )

    subprocess.run([
        "ffmpeg",
        "-y",
        "-i",
        "audio.ogg",
        "audio.wav"
    ])

    transcription = whisper_model.transcribe(
        "audio.wav"
    )

    question = transcription["text"].strip()

    response = generate_response(
        question
    )

    final_response = (
        f"Pergunta:\n{question}\n\n"
        f"Resposta:\n{response}"
    )

    await update.message.reply_text(
        final_response
    )

In [ ]:
app = ApplicationBuilder().token(
    TELEGRAM_TOKEN
).build()

app.add_handler(
    MessageHandler(
        filters.TEXT & ~filters.COMMAND,
        handle_text
    )
)

app.add_handler(
    MessageHandler(
        filters.VOICE,
        handle_voice
    )
)

print("Bot configurado")

In [ ]:
nest_asyncio.apply()

print("Bot online")

await app.initialize()
await app.start()
await app.updater.start_polling()